### 安装必要的库

- pycryptodome:提供标准的 AES 实现（用于验证结果）
- numpy:用于矩阵运算（AES 操作依赖大量的矩阵运算）
- matplotlib:用于可视化状态矩阵的变化
- ipywidgets:用于创建交互式滑块，动态观察加密过程

In [ ]:
!pip install numpy pycryptodome matplotlib ipywidgets

## AES 算法概述

#### 什么是 AES？  

高级加密标准（AES）是目前世界上最广泛使用的对称加密算法。

- 对称加密：加密和解密使用相同的密钥
- 分组密码：他不会一次性加密所有数据，而是吧数据分为固定大小的块。AES 的数据块大小固定为 128 位（16Byte）
- 密钥长度：AES 支持 128、192和256 位的密钥。本教程以 AES-128 为例。

#### AES 的核心思想：混淆与扩算

AES 内部是一个 4x4 的字节矩阵（称为 状态矩阵 State）。加密过程包含 10 轮循环操作，每一轮操作都包含四个步骤：

1. SubBytes（字节代换）- 混淆
2. ShiftRows（行移位）- 扩散
3. MixColumns（列混淆）- 扩散（最后一轮无此步骤）
4. AddRoundKey（轮密钥加）- 混入密钥

In [30]:
#在开始前，我们先定义一下如何把 16 字节的一维数组变成 4 X 4 的状态矩阵
def bytes_to_state(data):
    state = [[0] * 4 for _ in range(4)]
    for i, byte in enumerate(data):
        state[i%4][i//4] = byte
    return state

#再将 4 X 4 的状态矩阵还原回一维字节数组
def state_to_bytes(state):
    return bytes(state[r][c] for r in range(4) for c in range(4))

#进行一个测试
plaintext = b"Hello AES world!"
state = bytes_to_state(plaintext)
print("Init state:")
for row in state:
    print([hex(x) for x in row])

Init state:
['0x48', '0x6f', '0x53', '0x72']
['0x65', '0x20', '0x20', '0x6c']
['0x6c', '0x41', '0x77', '0x64']
['0x6c', '0x45', '0x6f', '0x21']


## 字节代换

原理：这是 AES 中唯一的非线性变换，也是破解 AES 最难跨越的屏障。它使用一个预先计算好的查找表，称为 S盒（S-Box）。状态矩阵中的每一个字节，都根据它的值去 S 盒中找到对应的新字节进行替换。
例如：当前字节为 0x53，程序会检查 S 盒中第五行第三列，将其替换为对应的值。

In [31]:
# 预先定义好的 AES S-Box (256个字节)
SBOX =[
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16
]

def sub_bytes(state):
    for r in range(4):
        for c in range(4):
            state[r][c] = SBOX[state[r][c]]
    return state

# 演示 SubBytes
state = sub_bytes(state)
print("SubBytes 后的矩阵:")
for row in state: print([hex(x) for x in row])

SubBytes 后的矩阵:
['0x52', '0xa8', '0xed', '0x40']
['0x4d', '0xb7', '0xb7', '0x50']
['0x50', '0x83', '0xf5', '0x43']
['0x50', '0x6e', '0xa8', '0xfd']


## 行移位
原理：  
为了打乱数据的排列，我们对状态矩阵的 4 行进行循环左移：
- 第 0 行：保持不变
- 第 1 行：循环左移 1 字节
- 第 2 行：循环左移 2 字节
- 第 3 行：循环左移 3 字节
这样做的目的是使得矩阵的每一列都包含了原来不同列的元素，为下一步做准备

## 实现 ShiftRows

In [32]:
def shift_rows(state):
    for i in range(len(state)):
        state[i] = state[i][i:] + state[i][:i]
    return state

#sample
state = shift_rows(state)
print("ShiftRow 后的矩阵：")
for row in state: print([hex(x) for x in row])

ShiftRow 后的矩阵：
['0x52', '0xa8', '0xed', '0x40']
['0xb7', '0xb7', '0x50', '0x4d']
['0xf5', '0x43', '0x50', '0x83']
['0xfd', '0x50', '0x6e', '0xa8']


## 列混淆
原理：
这一步是 AES 中最复杂的数学变换，它涉及到 伽罗瓦域 的矩阵乘法。你可以把它理解为对状态矩阵的每一列进行了一次复杂的基因重组。一列中的 4 个字节，经过运算后，每一个新字节都与原来的 4 个旧字节相关，这实现了深度的“扩散”效果。
> AES 最后一轮不会执行这一步，为了方便后续的解密

In [33]:
def xtime(a):
    return (((a << 1) ^ 0x1B) & 0xFF) if (a & 0x80) else (a << 1)

def mix_signle_column(a):
    # a 是包含 4 个字节的列表
    t = a[0] ^ a[1] ^ a[2] ^ a[3]
    u = a[0]
    a[0] ^= t ^ xtime(a[0]^a[1])
    a[1] ^= t ^ xtime(a[1]^a[2])
    a[2] ^= t ^ xtime(a[2]^a[3])
    a[3] ^= t ^ xtime(a[3]^u)
    return a

def mix_columns(state):
    for c in range(4):
        col = [state[r][c] for r in range(4)]
        col = mix_signle_column(col)
        for r in range(4):
            state[r][c] = col[r]
    return state

#smaple
state = mix_columns(state)
print("MixColumns 后的矩阵：")
for row in state:
    print([hex(x) for x in row])

MixColumns 后的矩阵：
['0x6e', '0x9a', '0xf', '0x7c']
['0xde', '0x48', '0xd3', '0xec']
['0x8', '0x69', '0xaf', '0xf3']
['0x55', '0xb7', '0xf0', '0x45']


## 轮密钥加与密钥扩展
原理：
到目前为止